# **Prédire le premier article ajouté dans un panier**

## Import

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler

## Preprocessing

In [2]:
df = pd.read_csv("./datasets-cleaned/df_order_products_clean.csv")

# On crée une colonne permettant de récupérer la taille du panier de chaque commande
df["cart_size"] = df.groupby("order_id")["product_name"].transform("count")

# On  récupère le nombre de vente de chaque produit
product_sales_nb = df["product_name"].value_counts()

# On garde uniquement les premiers produits ajoutés dans le panier
df_first_order = df[df["add_to_cart_order"] == 1].copy() #.copy() permet de forcer pandas à créer un nouveau dataframe et de ne pas seulement générer une visualisation

# On crée une colonne donnant le nombre de ventes total pour chaque article
df_first_order["product_sales_nb"] = df_first_order["product_name"].map(product_sales_nb)

# On récupère le nombre de fois qu'un article est acheté en premier
product_sales_in_first_nb = df_first_order["product_name"].value_counts()

# On crée une colonne indiquant le nombre de fois que l'article a été vendu en premier
df_first_order["product_sales_in_first_nb"] = df_first_order["product_name"].map(product_sales_in_first_nb)

# On crée une colonne qui indique la probabilité qu'un article soit achété en premier
df_first_order["sales_in_first_prob"] = (df_first_order["product_sales_in_first_nb"]/df_first_order["product_sales_nb"])

In [3]:
##############################################################
#################        Observations        #################
##############################################################

# Nombre de ventes de chaque produit
print(product_sales_nb.head())

# DataFrame de travail
print(df_first_order.head())
print(df_first_order.shape)

# Connaître le nombre de classes total
print("Nombre de produits différents ajoutés en 1er :", len(df_first_order["product_id"].unique()))

# Etudier un filtre à appliquer sur les classes pour pouvoir entrainer LightGBM et SGDClassifier
grouped_df_first_order = df_first_order.groupby("product_id").agg({
    "sales_in_first_prob": "first"
})
print("Nombre de produits avec plus de 10% de chance d'être commandé en 1er :", len(grouped_df_first_order[grouped_df_first_order["sales_in_first_prob"] > 0.1]))
print("Nombre de produits avec plus de 50% de chance d'être commandé en 1er :", len(grouped_df_first_order[grouped_df_first_order["sales_in_first_prob"] > 0.5]))

product_name
Banana                    187148
Bag of Organic Bananas    152420
Organic Strawberries      104957
Organic Baby Spinach       95561
Organic Hass Avocado       85569
Name: count, dtype: int64
    order_id  product_id  add_to_cart_order  reordered  user_id  order_number  \
0          2       33120                  1          1   202279             3   
9          4       46842                  1          0   178520            36   
22         6       40462                  1          0    22352             4   
25         8       23423                  1          1     3107             5   
26        21        5009                  1          1    88772            13   

    order_dow  order_hour_of_day  days_since_prior_order  \
0           5                  9                       8   
9           1                  9                       7   
22          1                 12                      30   
25          4                  6                      17   
26       

## KNNClassifier avec une prédictions parmi tous les 1er produits achetés

In [4]:
################################################################
##################        KNNClassifier       ##################
################################################################

feature_cols = [
    "reordered",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "cart_size",
    "product_sales_nb",
    "product_sales_in_first_nb",
    "sales_in_first_prob"
]

# On applique un get_dummies uniquement pour les colonnes contenant des valeurs catégorielles et non quantitatives
X = pd.get_dummies(df_first_order[feature_cols], columns=["order_dow"])

y = df_first_order["product_id"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = KNeighborsClassifier()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Five first predictions :{y_pred[:5]}")
print(f"Accuracy score :{accuracy}, le modèle a prédit correctement {round(accuracy*100,  2)}% des 1er produits.")
print(f"Precision score :{precision}, parmi les produits prédits, {round(precision*100,  2)}% étaient vraiment ajoutés en 1er.")
print(f"Recall score :{recall}, le modèle a détecté {round(recall*100,  2)}% des produits réellement ajoutés en 1er")
print(f"F1 score :{f1}")

Five first predictions :[21386 18389 28156   350 12254]
Accuracy score :0.7585513395229118, le modèle a prédit correctement 75.86% des 1er produits.
Precision score :0.1440622711966163, parmi les produits prédits, 14.41% étaient vraiment ajoutés en 1er.
Recall score :0.1421676213824833, le modèle a détecté 14.22% des produits réellement ajoutés en 1er
F1 score :0.1366081121860017


## Comparaison KNN, SGDClassifier et LightGBM avec seulement les articles ayant plus de 50% de chance d'être ajoutés en 1er

In [5]:
################################################################
##################        Preprocessing       ##################
################################################################

df_first_order = df_first_order[df_first_order["sales_in_first_prob"] > 0.5].copy()

In [6]:
################################################################
##################        KNNClassifier       ##################
################################################################

feature_cols = [
    "reordered",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "cart_size",
    "product_sales_nb",
    "product_sales_in_first_nb",
    "sales_in_first_prob"
]

# On applique un get_dummies uniquement pour les colonnes contenant des valeurs catégorielles et non quantitatives
X = pd.get_dummies(df_first_order[feature_cols], columns=["order_dow"])

y = df_first_order["product_id"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = KNeighborsClassifier()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Five first predictions :{y_pred[:5]}")
print(f"Accuracy score :{accuracy}, le modèle a prédit correctement {round(accuracy*100,  2)}% des 1er produits.")
print(f"Precision score :{precision}, parmi les produits prédits, {round(precision*100,  2)}% étaient vraiment ajoutés en 1er.")
print(f"Recall score :{recall}, le modèle a détecté {round(recall*100,  2)}% des produits réellement ajoutés en 1er")
print(f"F1 score :{f1}")

Five first predictions :[ 8610 45328  6209  2590  3287]
Accuracy score :0.2116788321167883, le modèle a prédit correctement 21.17% des 1er produits.
Precision score :0.06602779424356188, parmi les produits prédits, 6.6% étaient vraiment ajoutés en 1er.
Recall score :0.07252519264967397, le modèle a détecté 7.25% des produits réellement ajoutés en 1er
F1 score :0.06586568111464378


In [7]:
################################################################
##################        SGDClassifier       ##################
################################################################

feature_cols = [
    "reordered",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "cart_size",
    "product_sales_nb",
    "product_sales_in_first_nb",
    "sales_in_first_prob"
]

# On applique un get_dummies uniquement pour les colonnes contenant des valeurs catégorielles et non quantitatives
X = pd.get_dummies(df_first_order[feature_cols], columns=["order_dow"])

y = df_first_order["product_id"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = SGDClassifier()

# On normalise les valeurs pour donner à chaque feature le même poids d'importance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Five first predictions :{y_pred[:5]}")
print(f"Accuracy score :{accuracy}, le modèle a prédit correctement {round(accuracy*100,  2)}% des 1er produits.")
print(f"Precision score :{precision}, parmi les produits prédits, {round(precision*100,  2)}% étaient vraiment ajoutés en 1er.")
print(f"Recall score :{recall}, le modèle a détecté {round(recall*100,  2)}% des produits réellement ajoutés en 1er")
print(f"F1 score :{f1}")

Five first predictions :[41790 13404  6209  9711 35204]
Accuracy score :0.13503649635036497, le modèle a prédit correctement 13.5% des 1er produits.
Precision score :0.039575392755715644, parmi les produits prédits, 3.96% étaient vraiment ajoutés en 1er.
Recall score :0.046288209606986895, le modèle a détecté 4.63% des produits réellement ajoutés en 1er
F1 score :0.03676227338891094


In [8]:
#################################################################
####################          LightGBM       ####################
#################################################################

feature_cols = [
    "reordered",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "cart_size",
    "product_sales_nb",
    "product_sales_in_first_nb",
    "sales_in_first_prob"
]

# Pas de get_dummies car lightGBM gère les variables catégorielles
X = df_first_order[feature_cols]

y = df_first_order["product_id"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = lgb.LGBMClassifier(
    objective="multiclass",  # type de problème : classification multi-classe
    num_class=424,           # nombre total de classes possibles (produits)
    n_estimators=100,        # nombre d'itérations de boosting (= arbres ajoutés)
    learning_rate=0.05,      # poids appliqué à chaque nouvel arbre
    n_jobs=-1,               # nombre de threads CPU utilisés (-1 = tous)
    verbosity=-1             # désactive les logs dans la console
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Five first predictions :{y_pred[:5]}")
print(f"Accuracy score :{accuracy}, le modèle a prédit correctement {round(accuracy*100,  2)}% des 1er produits.")
print(f"Precision score :{precision}, parmi les produits prédits, {round(precision*100,  2)}% étaient vraiment ajoutés en 1er.")
print(f"Recall score :{recall}, le modèle a détecté {round(recall*100,  2)}% des produits réellement ajoutés en 1er")
print(f"F1 score :{f1}")

Five first predictions :[41790 45328  6209  4212 25358]
Accuracy score :0.4927007299270073, le modèle a prédit correctement 49.27% des 1er produits.
Precision score :0.1962468193384224, parmi les produits prédits, 19.62% étaient vraiment ajoutés en 1er.
Recall score :0.1984732824427481, le modèle a détecté 19.85% des produits réellement ajoutés en 1er
F1 score :0.19491094147582697
